<span style="float: left;padding: 1.3em">![logo](https://github.com/gw-odw/odw/blob/main/logo.png?raw=1)</span>

# Gravitational Wave Open Data Workshop

## Tutorial A.1: Parameter estimation on GW190828_063405 using GWOSC data and LALInference software.

This tutorial provides step-by-step instructions to estimate the properties of the source of the gravitational events (GW) detected by the LIGO-Virgo-KAGRA (LVK) collaboration. The parameter estimation is performed with `LALInference`, a software library developed by the LVK collaboration to perform Bayesian inference of GW source properties for binary systems of black holes and/or neutron stars, and used during the three first observation runs. This tutorial can be adapted to analyse any event in the GWTC-2 & 3 catalogs.

### Content
The tutorial provides instruction to:
- install `LALInference`
- download the GW strain
- download the PSD and calibration envelops
- read the configuration options
- start a `LALInference` run

### Note
This tutorial is intended to be ran with a local `conda` installation on a mac or Linux computer, and does not support mybinder nor Google Colab.
Instructions about setting up a `conda` environment are available on [this page](../../setup.md) (option 3).
   
### Documentation
- The formalism of the LALInference package is described in the following article:
[Phys. Rev. D 91, 042003 (2015)](https://arxiv.org/abs/1409.7215)
- A lecture of GW parameter estimation with Bayesian inference: [Recording of GWOSC ODW #2](https://www.youtube.com/watch?v=_NaAKeVJe1s&t=1s)
- All the GWOSC open data workshops lectures: [Link to GWOSC ODW](https://gwosc.org/workshops/)

### Acknowledgements
This tutorial is created by Leïla Haegel (leila.haegel@ligo.org), based on utilitary software designed by the LVK collaboration members. A special thanks is due to Charlie Hoy and Marc Arène for their assistance in understanding `PESummary` and `LALInference`.

**Last update**: May 2022


##  A bit of theory

**Introduction**: `LALInference` [1] uses Markov chains to sample the posterior probability of the GW source parameters (for GW emitted by binary systems of black holes and/or neutron stars).

**Markov chains**: Markov chains are semi-random walks in specific space, some Markov chain algorithms such as Metropolis-Hastings or Nested Sampling aim at having the semi-random walk steps be proportonal to a target distribution. In LALInference the target distribution is the joint posterior probability of the GW source parameters (such as mass, spin, binary localisation and orientation.

**Posterior probability**: The posterior probability is given by the Bayes theorem:

$$ p(\theta | d, H) = \frac{p(d|\theta,H) \ p(\theta|H)}{p(d|H)} $$

where $d$ are the data, $\theta$ are the GW parameters, and $H$ the hypotheses according to which the posterior probability is computed. $p(\theta | d, H)$ is the posterior probability, $p(\theta|H)$ is the prior probability, $p(d|H)$ is the evidence, and $p(d|\theta,H)$ is the likelihood given by [1]:

$$ p(d|\vec{\theta},H_S,S_n(f),\vec{\eta}) = exp \sum_i \left[ -\frac{2 |\tilde{h_i}(\vec{\theta}) - \tilde{d_i}|^2}{T \eta(f_i) S_n(f_i)} - \frac{1}{2} log(\pi \eta_i T S_n(f_i)/2 \right] $$

where $p(d|\vec{\theta},H_S,S_n(f),\vec{\eta})$ is the likelihood marginalized over the calibration envelops, $d_i$ is the strain and $S_n(f_i)$ is the Power Spectral Density (PSD). We therefore need all those quantities, that are described in details below.

**LALInference**: `LALInference` is a software library computing the posterior probability $p(\theta | d, H)$ with a Markov chain process. It returns the steps of the chains, i.e. the value of the posterior probability at different points of the parameter space. Their distributions enable to infer the most probable value and credible intervals of the GW source parameters. This tutorial downloads the necessary inputs and configures a `LALInference` run.


[1] [Phys. Rev. D 91, 042003 (2015)](https://arxiv.org/abs/1409.7215)


In [1]:
# Those 2 lines are just to avoid some harmless warnings when importing packages
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

## Installation (execute only if running on a cloud platform, like Google Colab, or if you haven't done the installation already!)

> ⚠️ **Warning**: restart the runtime after running the cell below.
>
> To do so, click "Runtime" in the menu and choose "Restart and run all".
>
> You may see error messages but installation usually works.
> If you experience problems, please [report an issue](https://github.com/gw-odw/odw/issues).

In [2]:
# -- Those packages will be needed for the execution
! pip install -q lalsuite==7.25 pesummary==1.6.1 'cryptography==43.0.0'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.0/115.0 kB 13.1 MB/s eta 0:00:00


## Initialization

We begin by importing some commonly used functions

In [3]:
import os
import shutil
import lal
import json
import numpy as np
from pesummary.gw import fetch
from pesummary.io import read
from pesummary.gw.file.calibration import CalibrationDict

Then we create the directory in which we will work for this tutorial. All the input and output files will be stored in `tutoA.1` (modify at your convenience if needed).

In [4]:
# get the local workdir directory name
cwd = os.getcwd()

# create a new directory called tutoA.1 for the run
workdir = cwd + '/tuto_A.1/'
os.makedirs(workdir, exist_ok=True)

## Getting the data: GW190828_063405

In this notebook, we analyse GW190828_063405. Information on the event can be found on the [GWOSC page](https://gwosc.org/eventapi/html/GWTC-2/GW190828_063405/).

The notebook can be adapted to analyse any other event from the [GWTC-2](https://gwosc.org/GWTC-2/), [2.1](https://gwosc.org/GWTC-2.1/) & [3](https://gwosc.org/GWTC-3/) catalogs.



In [5]:
evt = 'GW190828_063405'
catalog = 'GWTC-2'

### PESummary files

The PE files contains the PSD, calibration envelops and inference run configuration options, as well as the posterior samples from the `LALInference` runs performed by the LVK collaboration.

The file is retrieved and parsed using the `pesummary` package (more information on the [PESummary documentation](https://lscsoft.docs.ligo.org/pesummary/reference/)).





We start by downloading the PESummary files in the working directory (they will be saved in a directory named as the event):



In [6]:
fetch.fetch_open_samples(evt, catalog=catalog,
                         unpack=True, read_file=False, delete_on_exit=False,
                         outdir=workdir)

2026-05-10  00:53:27 PESummary WARNING : Multiple URLs found for GW190828_063405: https://dcc.ligo.org/public/0169/P2000223/005/GW190828_063405.tar, https://dcc.ligo.org/public/0169/P2000223/007/GW190828_063405.tar. Fetching most recent based on Unique IDs: https://dcc.ligo.org/public/0169/P2000223/007/GW190828_063405.tar


Extracting all files from /tmp/astropy-download-1128-ks0fgtx9


PosixPath('/content/tuto_A.1/GW190828_063405')

Then we open it, after checking that it is downloaded as expected:

In [7]:
pe_path = workdir + evt + '/' + evt + '.h5'

if os.path.exists(pe_path):
    print('Parameter estimation file found in: ' + pe_path + '\nOpening it.')
    pe_data = read(pe_path, package="gw")
else:
    print('Error: parameter estimation file not found in: ' + pe_path +
          '\n You need to download the PE file manually from the GWOSC: ' +
          'https://www.gwosc.org/eventapi/html/'+ catalog + '/' + evt)

Parameter estimation file found in: /content/tuto_A.1/GW190828_063405/GW190828_063405.h5
Opening it.


2026-05-10  00:53:44 PESummary WARNING : Unable to install 'pycbc'. You will not be able to use some of the inbuilt functions.
2026-05-10  00:53:44 PESummary WARNING : Unable to install 'pycbc'. You will not be able to use some of the inbuilt functions.


Several analyses are included in the PESummary file.
We will chose the analysis:
- starting with `C01` (release-level calibration)  
- using the `IMRPhenomPv2` approximant. The approximant refers to the model used to simulate the gravitational waveform for the analysis. It does not impact the extraction of the PSD nor the calibration file, but will impact the options chosen to run the parameter estimation later in the tutorial.

In [8]:
label = 'C01:IMRPhenomPv2'

if label in pe_data.labels:
    print(label + ' found in the list of analyses. Analyses available are: ', pe_data.labels)
else:
    print(label + ' not in the list of analyses. Analyses available are: ', pe_data.labels)



C01:IMRPhenomPv2 found in the list of analyses. Analyses available are:  ['C01:IMRPhenomD', 'C01:IMRPhenomPv2', 'C01:NRSur7dq4', 'C01:SEOBNRv4P', 'C01:SEOBNRv4PHM', 'PrecessingSpinIMR', 'PrecessingSpinIMRHM', 'PublicationSamples', 'ZeroSpinIMR']


### PSD files

The Power Spectral Density (PSD) is the distribution of the signal power in the frequency domain. The PSD can be obtained from the PE samples file using `pesummary`. The PSD must be downloaded with 1 file / interferometer as done below (any directory can be specified, working directory used in this example).



In [9]:
psds = pe_data.psd[label]
ifos = psds.detectors

psd_files = {}

for ifo in ifos:
    freq = psds[ifo].frequencies
    psd  = psds[ifo].strains

    psd_path = workdir + evt + '_PSDs_' + ifo + '.dat'
    psd_files[ifo] = psd_path

    np.savetxt(psd_path , np.column_stack([freq, psd]), delimiter='\t')

    print('PSD file downloaded in: ', psd_path)



PSD file downloaded in:  /content/tuto_A.1/GW190828_063405_PSDs_H1.dat
PSD file downloaded in:  /content/tuto_A.1/GW190828_063405_PSDs_L1.dat
PSD file downloaded in:  /content/tuto_A.1/GW190828_063405_PSDs_V1.dat


### Calibration envelops files

The calibration envelops represent the uncertainty due to the detector calibration. The files can be obtained from the PE samples file using `pesummary`. They must be downloaded with 1 file / interferometer as done below (any directory can be specified, working directory used in this example).

In [10]:
calib_envs = pe_data.priors['calibration'][label]
calib_data = CalibrationDict(calib_envs)
ifos = calib_data.detectors

calib_files = {}

for ifo in ifos:

    freq = calib_data[ifo].frequencies
    mag  = calib_data[ifo].magnitude
    phi  = calib_data[ifo].phase
    mag_m1s = calib_data[ifo].magnitude_lower
    phi_m1s = calib_data[ifo].phase_lower
    mag_p1s = calib_data[ifo].magnitude_upper
    phi_p1s = calib_data[ifo].phase_upper

    calib_path = workdir + evt + '_CalEnv_' + ifo + '.txt'
    calib_files[ifo] = calib_path
    col_title = 'Frequency Median Mag Phase (Rad) -1 Sigma Mag -1 Sigma Phase +1 Sigma Mag +1 Sigma Phase'
    np.savetxt(calib_path, np.column_stack([freq, mag, phi, mag_m1s, phi_m1s, mag_p1s, phi_p1s]), delimiter=' ', header=col_title)

    print('Calibration file downloaded in: ', calib_path)



Calibration file downloaded in:  /content/tuto_A.1/GW190828_063405_CalEnv_H1.txt
Calibration file downloaded in:  /content/tuto_A.1/GW190828_063405_CalEnv_L1.txt
Calibration file downloaded in:  /content/tuto_A.1/GW190828_063405_CalEnv_V1.txt


### Strain files

The strain files are available on the GWOSC website and can be downloaded below using the `pesummary` package.
The functions options refer to the following signal properties:

**Duration**: Up to GWTC-3, all the gravitational waves detected from binary black hole systems have signal duration < 32 s.
For binary systems containing one or two neutron stars, the signal is longer and strain files of 4096 s must be used.

**Sampling rate**: The sampling rate can be determined by approximating that the maximal frequency $f_{max}$ to the frequency at the innermost stable circular orbit $f_{ISCO}$:  $ f_{max} \simeq f_{ISCO}  \simeq \frac{4.4 kHz}{M_{tot}} $

For a total mass $ M_{tot} > 4$ for binary black holes, we find the higher bound on the maximal frequency:
$ f_{max} \simeq 1048 Hz $

The sampling rate corresponds to the Nyquist frequency:
$f_{Nyquist} = 2 f_{max}  = 2048 Hz$

so the files with a sampling rate of $ f_{Nyquist} = 4096 Hz$ can be used for any binary system of black holes detected by the LVK.



In [11]:
strain_files = {}

for ifo in ifos:

    strain = fetch.fetch_open_strain(evt, catalog=catalog, IFO=ifo, duration=32, sampling_rate=4096., format='gwf',
                                     delete_on_exit=True, read_file=False, outdir=workdir)
    strain_files[ifo] = str(strain)


The strain must be passed to `LALInference` using cache files, created with a bash command:

In [12]:
strain_cache_files = {}

for ifo in ifos:
    strain_cache_files[ifo] = strain_files[ifo].split('.gwf')[0]+ '.lcf'
    command = 'ls ' + strain_files[ifo] +' | lal_path2cache > ' + strain_cache_files[ifo]
    print(command)
    os.system(command)

ls /content/tuto_A.1/H-H1_GWOSC_4KHZ_R1-1251009248-32.gwf | lal_path2cache > /content/tuto_A.1/H-H1_GWOSC_4KHZ_R1-1251009248-32.lcf
ls /content/tuto_A.1/L-L1_GWOSC_4KHZ_R1-1251009248-32.gwf | lal_path2cache > /content/tuto_A.1/L-L1_GWOSC_4KHZ_R1-1251009248-32.lcf
ls /content/tuto_A.1/V-V1_GWOSC_4KHZ_R1-1251009248-32.gwf | lal_path2cache > /content/tuto_A.1/V-V1_GWOSC_4KHZ_R1-1251009248-32.lcf


## Chose the LALInference options

The `LALInference` software performs parameter estimation with Bayesian inference, using Markov chain techniques. Two types of Markov chaim algorithms are implemented : Markov Chain Monte-Carlo (MCMC) and nested sampling. `LALInference` supports a wide range of options, from the input files specification to the prior to set up to the number of parallel inference runs to launch. All those options are stored in the config section of the `PESummary` files. In the notebook, we read the  `PESummary` file to create a bash file that will start the inference on our machine with the desired options.   

We will use the information in the sections below to set up our bash file:

In [13]:
pe_data.config[label].keys()

dict_keys(['analysis', 'bayeswave', 'condor', 'data', 'datafind', 'engine', 'input', 'lalinference', 'ligo-skymap-from-samples', 'ligo-skymap-plot', 'mpi', 'paths', 'resultspage', 'skyarea', 'statevector'])

### Create the bash file

We know create a bash file that we will write with the necessary options.

In [14]:
run_path = workdir + 'lalinference_run.sh'
run_path

'/content/tuto_A.1/lalinference_run.sh'

Then we chose the executable and relevant options from the `analysis` and `engine` sections of the config options.

The location of the executable(s) depends on the machine and the method you used to install the software.
In order to find them, we will use the `which` function (from the `shutil` package).

**For nested sampling**, we need to specify:
- the number of live points `nlive`
- the number of points `maxmcmc` for independent MCMC chains (optional)
- the tolerance on the evidence `tolerance` to obtain to stop the inference (optional)

In [15]:
if pe_data.config[label]['analysis']['engine'] == 'lalinferencenest':
    with open(run_path, "w") as run_file:
        print(shutil.which("lalinference_nest") + ' \\', file=run_file)
        print('--nlive ' + pe_data.config[label]['engine']['nlive'] + ' \\', file=run_file)
        if 'maxmcmc' in pe_data.config[label]['engine']:
            print('--maxmcmc ' + pe_data.config[label]['engine']['maxmcmc'] + ' \\', file=run_file)
        if 'tolerance' in pe_data.config[label]['engine']:
            print('--tolerance ' + pe_data.config[label]['engine']['tolerance'] + ' \\', file=run_file)



**For MCMC**, we need to specify:
- the path to the MPI executable for parallelisation
- the number of parallel jobs `np`
- the number of independent samples `neff` to obtain to stop the inference (optional)
- the number of temperature chains `ntemps` (optional)
- the option to adapt the temperature chains `adapt-temps` (optional)


In [16]:
if pe_data.config[label]['analysis']['engine'] == 'lalinferencemcmc':
    with open(run_path, "w") as run_file:
        print(shutil.which("lalinference_mpi_wrapper") + ' \\', file=run_file)
        print('--mpirun ' + shutil.which("mpirun") + ' \\', file=run_file)
        print('--executable ' + shutil.which("lalinference_mcmc") + ' \\', file=run_file)
        print('--np ' + pe_data.config[label]['engine']['np'] + ' \\', file=run_file)

        if 'neff' in pe_data.config[label]['engine']:
            print('--neff ' + pe_data.config[label]['engine']['neff'] + ' \\', file=run_file)
        if 'ntemps' in pe_data.config[label]['engine']:
            print('--ntemps ' + pe_data.config[label]['engine']['ntemps'] + ' \\', file=run_file)
        if 'adapt-temps' in pe_data.config[label]['engine']:
            print('--adapt-temps \\', file=run_file)


### Specify the input files

Specify the location of the input files for each interferometer, with the initial frequency extracted from the `lalinference` section of the configuration options.


In [17]:
if 'H1' in ifos:
    with open(run_path, "a") as run_file:
        print('--ifo H1 \\', file=run_file)
        print('--H1-cache ' + strain_cache_files['H1'] + ' \\', file=run_file)
        print('--H1-channel H1:GWOSC-4KHZ_R1_STRAIN \\', file=run_file)
        if 'flow' in pe_data.config[label]['lalinference']:
            f_low = pe_data.config[label]['lalinference']['flow'].replace('\'', '\"')
            print('--H1-flow ' + str(json.loads(f_low)['H1']) + ' \\', file=run_file)
        print('--H1-psd ' + psd_files['H1'] + ' \\', file=run_file)
        print('--H1-spcal-envelope ' + calib_files['H1'] + ' \\', file=run_file)


In [18]:
if 'L1' in ifos:
    with open(run_path, "a") as run_file:
        print('--ifo L1 \\', file=run_file)
        print('--L1-cache ' + strain_cache_files['L1'] + ' \\', file=run_file)
        print('--L1-channel L1:GWOSC-4KHZ_R1_STRAIN \\', file=run_file)
        if 'flow' in pe_data.config[label]['lalinference']:
            f_low = pe_data.config[label]['lalinference']['flow'].replace('\'', '\"')
            print('--L1-flow ' + str(json.loads(f_low)['L1']) + ' \\', file=run_file)
        print('--L1-psd ' + psd_files['L1'] + ' \\', file=run_file)
        print('--L1-spcal-envelope ' + calib_files['L1'] + ' \\', file=run_file)


In [19]:
if 'V1' in ifos:
    with open(run_path, "a") as run_file:
        print('--ifo V1 \\', file=run_file)
        print('--V1-cache ' + strain_cache_files['V1'] + ' \\', file=run_file)
        print('--V1-channel V1:GWOSC-4KHZ_R1_STRAIN \\', file=run_file)
        if 'flow' in pe_data.config[label]['lalinference']:
            f_low = pe_data.config[label]['lalinference']['flow'].replace('\'', '\"')
            print('--V1-flow ' + str(json.loads(f_low)['V1']) + ' \\', file=run_file)
        print('--V1-psd ' + psd_files['V1'] + ' \\', file=run_file)
        print('--V1-spcal-envelope ' + calib_files['V1'] + ' \\', file=run_file)


### Specify the calibration

Chose if the calibration envelops are sampled during the run, and specify the number of nodes from the `engine` section of the configuration options.

In [20]:
if 'enable-spline-calibration' in pe_data.config[label]['engine']:
    with open(run_path, "a") as run_file:
        print('--enable-spline-calibration \\', file=run_file)

if 'spcal-nodes' in pe_data.config[label]['engine']:
    with open(run_path, "a") as run_file:
        print('--spcal-nodes ' + pe_data.config[label]['engine']['spcal-nodes'] + ' \\', file=run_file)


### Specify the signal options

The options are extracted from the `engine` section of the configuration options, including:
- the length of the data segment to analyse `seglen` (in seconds)
- the sampling rate `srate` (in Herts)
- the waveform approximant `approx`
- the reference frequency `fref` (optional)

The trigger time `trigtime` that must be copied from the [GWOSC page](https://gwosc.org/eventapi/html/GWTC-2/GW190828_063405/).

In [21]:
trig_time = 1251009263.8

In [22]:
with open(run_path, "a") as run_file:
    print('--trigtime ' + str(trig_time) + ' \\', file=run_file)
    print('--srate ' + pe_data.config[label]['engine']['srate'] + ' \\', file=run_file)
    print('--seglen ' + pe_data.config[label]['engine']['seglen'] + ' \\', file=run_file)
    print('--approx ' + pe_data.config[label]['engine']['approx'] + ' \\', file=run_file)

if 'fref' in pe_data.config[label]['engine']:
    with open(run_path, "a") as run_file:
        print('--fref ' + pe_data.config[label]['engine']['fref'] + ' \\', file=run_file)


### Specify the priors

The priors are specified in the `engine` section of the configuration options, with the bounds of syntax: `[param]-min` and `[param]-max`.

In [23]:
for key in pe_data.config[label]['engine']:
    if '-min'in key:
        with open(run_path, "a") as run_file:
            print('--' + key + ' ' + pe_data.config[label]['engine'][key] + ' \\', file=run_file)
    if '-max'in key:
        with open(run_path, "a") as run_file:
            print('--' + key + ' ' + pe_data.config[label]['engine'][key] + ' \\', file=run_file)



### Specify the output options

The posterior samples will be saved in `lalinference_samples.hdf5` and we will specify options to resume the inference if interrupted and to print the progress into `lalinference.out` and `lalinference.err` files.

In [24]:
posterior_path = workdir + 'lalinference_samples.hdf5'
out_path = workdir + 'lalinference.out'
err_path = workdir + 'lalinference.err'


In [25]:
with open(run_path, "a") as run_file:
    print('--outfile ' + posterior_path + ' \\', file=run_file)
    print('--resume \\', file=run_file)
    print('--progress \\', file=run_file)
    print('> ' + out_path + ' 2> ' + err_path + ' &', file=run_file, end="")


## Start the inference

We will now execute the bash file with a bash command. It is executed from the notebook, but can be started from the terminal as well (in the working directory). The inference process is long and often requires several days to be completed.

In [26]:
command = 'bash ' + run_path

print('Command: ' + command)

# Since the inference process is long and takes several days to be completed, you may not want to run it on the computer you are using now.
# os.system(command)

Command: bash /content/tuto_A.1/lalinference_run.sh


In [39]:
! bash /content/tuto_A.1/lalinference_run.sh